### Model achitecture and hyperparameter tuning 
No we are ready to define embeddings model architecture and perform hyperparameter tuning using maggy.
![Training Dataset](./images/maggy_hp.png)

We are going to use StellarGraph library to compute node embeddings. StellarGraph supports loading data via Pandas DataFrames, NumPy arrays, Neo4j and NetworkX graphs. 

---
**NOTE**:

Loading large scale dataset in to StellarGraph for training can not be handled with above mentioned fameworks. It will require loading data using frameworks such as `tf.data`. 

If your training datasets measure from couple of GB to 100s of GBs or even TBs contact us at Logical Clocks and we will help you to setup distributed training pipelines. 

---

## Install required libraries. Such as StellarGraph. 
##### To do this 1st navigate to python environment then install library by name and version
![Incremental Feature Engineering](./images/intall_python_lib.gif)

## Define hyperparameter searchspace for maggy

In [1]:
# Local replacement for Maggy Searchspace
# Using node2vec library instead of StellarGraph (which requires Python < 3.9)

import itertools
import os
import pandas as pd
import numpy as np
import networkx as nx
from node2vec import Node2Vec
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Define hyperparameter search space
search_space = {
    'walk_number': [2, 3],      # num_walks in node2vec
    'walk_length': [2, 3],      # walk_length in node2vec
    'emb_size': [16, 32]        # dimensions in node2vec
}

# Generate all combinations
param_combinations = list(itertools.product(
    search_space['walk_number'],
    search_space['walk_length'],
    search_space['emb_size']
))

print(f"Total hyperparameter combinations: {len(param_combinations)}")
print("Combinations:", param_combinations)

# Define paths
BASE_PATH = os.path.dirname(os.path.abspath("__file__"))
TRAINING_DATA_PATH = os.path.join(BASE_PATH, "training_data")
RESOURCES_PATH = os.path.join(BASE_PATH, "Resources")
os.makedirs(RESOURCES_PATH, exist_ok=True)

Total hyperparameter combinations: 8
Combinations: [(2, 2, 16), (2, 2, 32), (2, 3, 16), (2, 3, 32), (3, 2, 16), (3, 2, 32), (3, 3, 16), (3, 3, 32)]


/opt/miniconda/envs/amlgan/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Define hopsworks experiments wrapper function and put all the training logic there. 

In [2]:
def embeddings_computer(walk_number, walk_length, emb_size, G, alert_nodes_df):
    """
    Compute node embeddings using Node2Vec algorithm.
    Uses node2vec library (works with Python 3.10+).
    
    Returns accuracy of predicting is_sar from embeddings.
    """
    print(f'Computing embeddings (walk_number={walk_number}, walk_length={walk_length}, emb_size={emb_size})')
    
    # Create Node2Vec model
    node2vec = Node2Vec(
        G, 
        dimensions=emb_size, 
        walk_length=walk_length, 
        num_walks=walk_number,
        p=0.5,  # Return parameter
        q=2.0,  # In-out parameter
        workers=4,
        quiet=True
    )
    
    # Train word2vec model on random walks
    print('Training Node2Vec model...')
    model = node2vec.fit(window=10, min_count=1, batch_words=4)
    
    # Get embeddings for nodes that are in alert_nodes_df
    print('Extracting embeddings...')
    embeddings = []
    labels = []
    valid_nodes = []
    
    for _, row in alert_nodes_df.iterrows():
        node_id = str(row['id'])
        if node_id in model.wv:
            embeddings.append(model.wv[node_id])
            labels.append(row['is_sar'])
            valid_nodes.append(node_id)
    
    if len(embeddings) < 10:
        print(f"Warning: Only {len(embeddings)} nodes found in embeddings")
        return 0.5, model, None
    
    X = np.array(embeddings)
    y = np.array(labels)
    
    # Train a simple classifier to evaluate embedding quality
    print('Training classifier to evaluate embeddings...')
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    
    clf = LogisticRegression(max_iter=1000, random_state=42)
    clf.fit(X_train, y_train)
    
    y_pred = clf.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    
    print(f'Embedding evaluation accuracy: {accuracy:.4f}')
    
    return accuracy, model, clf

## Use above experiments wrapper function to conduct maggy hyperparameter search experiments.

In [3]:
# Local hyperparameter search (replaces Maggy experiment)
import json
import random

# Load training data from local CSV files
print("Loading training data...")
node_pdf = pd.read_csv(os.path.join(TRAINING_DATA_PATH, "node_td.csv"))
edge_pdf = pd.read_csv(os.path.join(TRAINING_DATA_PATH, "edges_td.csv"))
alert_nodes_df = pd.read_csv(os.path.join(TRAINING_DATA_PATH, "alert_nodes_td.csv"))

print(f"Nodes: {len(node_pdf)}, Edges: {len(edge_pdf)}, Alert nodes: {len(alert_nodes_df)}")

# Build NetworkX graph
print("\nBuilding NetworkX directed graph...")
G = nx.from_pandas_edgelist(
    edge_pdf, 
    source='source', 
    target='target', 
    edge_attr=['tx_type', 'base_amt'],
    create_using=nx.DiGraph()
)
print(f"Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

# Run hyperparameter search
results = []
best_accuracy = 0
best_hp = None
best_model = None

# Limit trials for faster execution
num_trials = 2  # Same as original Maggy config

random.seed(42)
trial_combinations = random.sample(param_combinations, min(num_trials, len(param_combinations)))

print(f"\nRunning {len(trial_combinations)} trials...")
print("=" * 50)

for i, (walk_number, walk_length, emb_size) in enumerate(trial_combinations):
    print(f"\n--- Trial {i+1}/{len(trial_combinations)} ---")
    hp = {'walk_number': walk_number, 'walk_length': walk_length, 'emb_size': emb_size}
    print(f"Hyperparameters: {hp}")
    
    try:
        accuracy, model, clf = embeddings_computer(
            walk_number, walk_length, emb_size, G, alert_nodes_df
        )
        results.append({'hp': hp, 'accuracy': accuracy})
        
        if accuracy > best_accuracy:
            best_accuracy = accuracy
            best_hp = hp
            best_model = model
    except Exception as e:
        print(f"Trial failed: {e}")
        import traceback
        traceback.print_exc()
        results.append({'hp': hp, 'accuracy': None, 'error': str(e)})

print("\n" + "=" * 50)
print("------ Hyperparameter Search Results ------")
print(f"BEST combination {best_hp} -- metric {best_accuracy:.4f}")
if len(results) > 1:
    valid_results = [r for r in results if r['accuracy'] is not None]
    if valid_results:
        accuracies = [r['accuracy'] for r in valid_results]
        print(f"WORST combination -- metric {min(accuracies):.4f}")
        print(f"AVERAGE metric -- {sum(accuracies)/len(accuracies):.4f}")
print(f"Total trials: {len(results)}")

Loading training data...
Nodes: 7347, Edges: 438386, Alert nodes: 7347

Building NetworkX directed graph...
Graph: 7347 nodes, 17070 edges

Running 2 trials...

--- Trial 1/2 ---
Hyperparameters: {'walk_number': 2, 'walk_length': 2, 'emb_size': 32}
Computing embeddings (walk_number=2, walk_length=2, emb_size=32)
Training Node2Vec model...
Extracting embeddings...
Training classifier to evaluate embeddings...
Embedding evaluation accuracy: 0.8891

--- Trial 2/2 ---
Hyperparameters: {'walk_number': 2, 'walk_length': 2, 'emb_size': 16}
Computing embeddings (walk_number=2, walk_length=2, emb_size=16)
Training Node2Vec model...
Extracting embeddings...
Training classifier to evaluate embeddings...
Embedding evaluation accuracy: 0.8891

------ Hyperparameter Search Results ------
BEST combination {'walk_number': 2, 'walk_length': 2, 'emb_size': 32} -- metric 0.8891
WORST combination -- metric 0.8891
AVERAGE metric -- 0.8891
Total trials: 2


## Output best hyperparameter as a json file, so we can use it next step

In [4]:
# Save best hyperparameters locally (replaces hops.hdfs.dump)
import json

EMBEDDINGS_HYPERPARAMS_FILE = 'embeddings_best_hp.json'
hp_path = os.path.join(RESOURCES_PATH, EMBEDDINGS_HYPERPARAMS_FILE)

with open(hp_path, 'w') as f:
    json.dump(best_hp, f, indent=2)

print(f"Saved best hyperparameters to: {hp_path}")
print(f"Best HP: {best_hp}")

Saved best hyperparameters to: /home/adnoman/projects/aml_gan/AMLend2end/Resources/embeddings_best_hp.json
Best HP: {'walk_number': 2, 'walk_length': 2, 'emb_size': 32}


### Managing experiments
Experiments service provides a unified view of all the experiments run using the `experiment` module.
<br>
As demonstrated in the gif it provides general information about the experiment and the resulting metric. Experiments can be visualized meanwhile or after training in a TensorBoard.
<br>
<br>
![Image7-Monitor.png](./images/experiments.gif)